# HASH Health QA — fine-tune a generator for the Ghana configs (Colab GPU)

Retrieval is hard-capped on Aka_Gha / Eng_Gha (answers ~99% unique, oracle@20 ≈ 0.34–0.39).
This notebook fine-tunes **mT5-base** (Apache-2.0) on those configs to generate bespoke answers,
scores them with our exact local metric, and exports Val + Test predictions to merge back home.

**Run cells in order:** 1 GPU → 2 install → 3 upload → 3b pre-flight → 4 train → 4d plots → 5 download.

**To beat (Val mean-ROUGE):** Aka_Gha ≈ 0.356, Eng_Gha ≈ 0.332 (our retrieval+hybrid).
Generation is merged at home only if it beats these per subset.

In [ ]:
# 1. Confirm a GPU is attached (Runtime > Change runtime type > GPU)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Install open-licensed deps (PINNED for exact reproducibility — Zindi top-10 req)
!pip -q install transformers==4.44.2 datasets==2.21.0 accelerate==0.34.2 peft==0.12.0 \
    sentencepiece==0.2.0 rouge-score==0.1.2 2>/dev/null
import transformers, peft, torch
print('transformers', transformers.__version__, '| peft', peft.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
# 3. Upload Train.csv, Val.csv, Test.csv AND train_generator.py (from QA_Summative_ML/src/).
#    Select ALL FOUR files in the dialog.
import os
os.makedirs('data', exist_ok=True)
from google.colab import files
print('Select: Train.csv, Val.csv, Test.csv, train_generator.py')
up = files.upload()
for name in up:
    if name.endswith('.csv'):
        os.replace(name, f'data/{name}')
print('data/:', os.listdir('data'))
print('train_generator.py uploaded:', os.path.exists('train_generator.py'))

In [ ]:
# 3b. PRE-FLIGHT CHECK — verifies all inputs exist AND that train_generator.py is the
#     current version (must support --run-name / --use-lora / --grad-ckpt). Stops early otherwise.
import os
need = ['train_generator.py', 'data/Train.csv', 'data/Val.csv', 'data/Test.csv']
missing = [f for f in need if not os.path.exists(f)]
if missing:
    print('❌ MISSING:', missing)
    print('cwd files:', [f for f in os.listdir('.') if not f.startswith('.')])
    raise SystemExit('Re-run cell 3 and select train_generator.py + all 3 CSVs.')
src = open('train_generator.py').read()
stale = [flag for flag in ('--run-name', '--use-lora', '--grad-ckpt') if flag not in src]
if stale:
    raise SystemExit(f'❌ train_generator.py is OUT OF DATE (missing {stale}). '
                     'Re-run cell 3 and upload the current src/train_generator.py.')
print('✅ all inputs present + train_generator.py is current — ready to train')

*(Alternative to cell 3: `from google.colab import drive; drive.mount('/content/drive')` and point `--data-dir` at your Drive folder. You still need `train_generator.py` in the working dir.)*

In [ ]:
# 4. Fine-tuning experiments. T4-friendly FAST config: mT5-SMALL, no gradient-checkpointing,
#    batch 16, short sequences (input 96 / target 160), 2 beams. Each run ~8-12 min on a T4.
#    (mT5-base full FT OOMs a 16GB T4 in fp32 — see 4c for base via LoRA.)
import os; os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# 4a. mT5-small, FULL fine-tune  (primary generator)
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python train_generator.py \
    --data-dir data --output-dir out --run-name mt5small_full \
    --model-name google/mt5-small --subsets Aka_Gha,Eng_Gha \
    --epochs 2 --train-bs 16 --grad-accum 1 --lr 1e-3 \
    --max-input 96 --max-target 160 --gen-len 160 --beams 2 --seed 42

# 4b. mT5-small, LoRA / PEFT  (parameter-efficient comparison)
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python train_generator.py \
    --data-dir data --output-dir out --run-name mt5small_lora \
    --model-name google/mt5-small --subsets Aka_Gha,Eng_Gha --use-lora \
    --epochs 2 --train-bs 16 --grad-accum 1 --lr 5e-4 \
    --max-input 96 --max-target 160 --gen-len 160 --beams 2 --seed 42

# 4c. (optional, ~45-60 min) mT5-BASE via LoRA — bigger/stronger, needs --grad-ckpt to fit a T4.
#     Its loss dropped well in testing (18->6) so it's the quality upgrade if time allows.
# !PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python train_generator.py \
#     --data-dir data --output-dir out --run-name mt5base_lora \
#     --model-name google/mt5-base --subsets Aka_Gha,Eng_Gha --use-lora --grad-ckpt \
#     --epochs 2 --train-bs 4 --grad-accum 4 --lr 5e-4 \
#     --max-input 96 --max-target 160 --gen-len 160 --beams 2 --seed 42

In [ ]:
# 4d. Learning curves + experiment comparison (REQUIRED: training/validation curves)
import os, json, glob
import pandas as pd, matplotlib.pyplot as plt

runs = [d for d in glob.glob('out/*') if os.path.isdir(d) and os.path.exists(f'{d}/history.csv')]
assert runs, 'No runs found in out/ — run cell 4 (training) first.'
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
rows = []
for d in sorted(runs):
    name = os.path.basename(d)
    h = pd.read_csv(f'{d}/history.csv')
    tr = h.dropna(subset=['train_loss']) if 'train_loss' in h else pd.DataFrame()
    ev = h.dropna(subset=['eval_loss']) if 'eval_loss' in h else pd.DataFrame()
    if len(tr): ax[0].plot(tr['step'], tr['train_loss'], marker='.', label=f'{name} train')
    if len(ev): ax[0].plot(ev['step'], ev['eval_loss'], marker='o', ls='--', label=f'{name} eval')
    m = json.load(open(f'{d}/metrics.json'))
    for s, v in m['per_subset'].items():
        rows.append({'run': name, 'subset': s, 'gen': v['gen_mean_rouge'], 'retrieval': v['retrieval_baseline']})
ax[0].set(title='Learning curves (train vs eval loss)', xlabel='step', ylabel='loss'); ax[0].legend(fontsize=8)

cmp = pd.DataFrame(rows)
piv = cmp.dropna(subset=['gen']).reset_index(drop=True)
piv['label'] = piv['run'] + '\n' + piv['subset']
x = range(len(piv)); w = 0.4
ax[1].bar([i - w/2 for i in x], piv['gen'], w, label='generation')
ax[1].bar([i + w/2 for i in x], piv['retrieval'], w, label='retrieval baseline')
ax[1].set_xticks(list(x)); ax[1].set_xticklabels(piv['label'], fontsize=7)
ax[1].set(title='Generation vs retrieval (Val mean-ROUGE)', ylabel='mean-ROUGE'); ax[1].legend()
plt.tight_layout(); plt.savefig('out/learning_and_comparison.png', dpi=120); plt.show()
print(cmp.to_string(index=False))

In [ ]:
# 5. Bundle all run outputs and download (robust — reports clearly if training didn't run)
import os, glob
from google.colab import files
runs = sorted(glob.glob('out/*/metrics.json'))
if not runs:
    print('❌ No outputs in out/. Training (cell 4) did not complete.')
    print('   cwd files     :', [f for f in os.listdir('.') if not f.startswith('.')])
    print('   train_generator.py present:', os.path.exists('train_generator.py'))
    print('   data/         :', os.listdir('data') if os.path.isdir('data') else 'MISSING')
    print('   -> run cell 3b (pre-flight), then re-run cell 4a and read its FIRST error line.')
else:
    print('✅ runs found:', [os.path.basename(os.path.dirname(r)) for r in runs])
    !cd out && zip -qr ../generator_outputs.zip */val_gen.csv */test_gen.csv */metrics.json */history.csv 2>/dev/null
    if os.path.exists('out/learning_and_comparison.png'):
        !zip -qj generator_outputs.zip out/learning_and_comparison.png 2>/dev/null
    files.download('generator_outputs.zip')
    print('Pick the run with the highest Val gen mean-ROUGE per subset; merge at home with make_submission_v5.py')

## Next steps at home
1. Unzip; put the best run's `val_gen.csv` + `test_gen.csv` in `QA_Summative_ML/outputs/generator/`.
2. Run `python src/make_submission_v5.py --gen-dir outputs/generator` — it Val-gates generation per
   subset (only replaces a Ghana answer if generation beats retrieval on Val) and writes `submission_v5.csv`.
3. If a subset improves locally, spend ONE Zindi probe to confirm public transfer.

**If mT5 underperforms:** enable 4c (`afriteva_v2_base`, better Akan — verify its licence first),
or try `--epochs 6 --lr 5e-4`, or train per-subset (one `--subsets` each).